In [ ]:
import os, pathlib, sys

root = pathlib.Path(os.getcwd())
while not (root / 'requirements.txt').exists():
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print(f'Working directory set to: {os.getcwd()}')

Working directory set to: d:\Jeet\projects\Data_Science\Project\Project_Medical_Image_Classification


In [ ]:
import torch, torch.nn as nn
from ml.training.model import MODEL_REGISTRY
from ml.training.dataloader import get_dataloaders

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


In [ ]:
train_loader, val_loader, _ = get_dataloaders(
    data_dir='ml/data/processed', batch_size=32, num_workers=0)

In [ ]:
model = MODEL_REGISTRY['densenet121'](pretrained=True, freeze_layers=True).to(device)

In [ ]:
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total:,} | Trainable: {trainable:,} ({100*trainable/total:.1f}%)')

Total params: 7,216,770 | Trainable: 2,423,042 (33.6%)


In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
model.train()
for i, (imgs, labels) in enumerate(train_loader):
    imgs, labels = imgs.to(device), labels.to(device)
    optimizer.zero_grad()
    outputs = model(imgs)
    loss = criterion(outputs, labels)
    loss.backward(); optimizer.step()
    if i % 20 == 0:
        print(f'Batch {i}/{len(train_loader)}  Loss: {loss.item():.4f}')


d:\Jeet\projects\Data_Science\Project\Project_Medical_Image_Classification\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Batch 0/163  Loss: 0.1004
Batch 20/163  Loss: 0.0922
Batch 40/163  Loss: 0.0409
Batch 60/163  Loss: 0.1529
Batch 80/163  Loss: 0.0515
Batch 100/163  Loss: 0.1226
Batch 120/163  Loss: 0.1980
Batch 140/163  Loss: 0.1852
Batch 160/163  Loss: 0.0232
